# Jour 1 : Système déductif vs. système inductif

Nous allons essayer de définir un algorithme permettant de classer un épigramme en "parle d'animal" ou non avec les deux approches.

In [ ]:
!pip install wget pandas


In [5]:
import pandas as pd
import os

try:
    data = pd.read_csv(f"{os.getcwd()}/corpus/epigrammes_anthologia_graeca_fr.csv")
except FileNotFoundError:
    !wget -P corpus https://raw.githubusercontent.com/alexiaschn/dhsi-2026/main/notebooks/corpus/epigrammes_anthologia_graeca_fr.csv
    data = pd.read_csv(f"{os.getcwd()}/corpus/epigrammes_anthologia_graeca_fr.csv")


data.head()

--2026-05-28 14:20:19--  https://raw.githubusercontent.com/alexiaschn/dhsi-2026/main/notebooks/corpus/epigrammes_anthologia_graeca_fr.csv
Résolution de raw.githubusercontent.com (raw.githubusercontent.com)… 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connexion à raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443… connecté.
requête HTTP transmise, en attente de la réponse… 200 OK
Taille : 5597 (5,5K) [text/plain]
Enregistre : ‘corpus/epigrammes_anthologia_graeca_fr.csv.2’

epigrammes_antholog 100%[===================>]   5,47K  --.-KB/s    ds 0s      

2026-05-28 14:20:19 (63,5 MB/s) - ‘corpus/epigrammes_anthologia_graeca_fr.csv.2’ enregistré [5597/5597]



,epigramme_id,book,fragment,urn,author_fr,text_fr
0,7.114,7,114,urn:cts:greekLit:tlg7000.tlg001.ag:7.114,Diogène Laërce,"Tu voulais, Héraclide, laisser aux hommes le b..."
1,7.115,7,115,urn:cts:greekLit:tlg7000.tlg001.ag:7.115,Diogène Laërce,"« Tu as vécu en chien, Antisthène. - C'est que..."
2,7.116,7,116,urn:cts:greekLit:tlg7000.tlg001.ag:7.116,Diogène Laërce,"« Diogène, allons, dis-moi quel destin t'a rav..."
3,7.120,7,120,urn:cts:greekLit:tlg7000.tlg001.ag:7.120,Xénophane,On dit qu'un jour il passait devant un jeune c...
4,7.171,7,171,urn:cts:greekLit:tlg7000.tlg001.ag:7.171,Mnasalcès,Ici aussi l'oiseau sacré reposera son aile rap...


## Système expert

Système à base de règle.

In [ ]:
import re

# définition de règle : motif textuel de base

motif_animal = re.compile(r"(chien|chat|souris|rat|pieuvre|marmotte)", flags=re.IGNORECASE)

def contient_animal(texte):
    if re.search(motif_animal, texte):
        print(texte)
        return True
    else:
        return False

for texte in data['text_fr']:
    print(contient_animal(texte))



Comment pourrions-nous améliorer ce programme ?

In [ ]:
# possible version améliorée du système expert

# récupération d'une liste plus complète de https://liste-animaux.fr/
import wget

# on télécharge la page web, elle s'enregistre automatiquement dans le dossier où se trouve le notebook
page_html = wget.download("https://liste-animaux.fr/")

# import du module BeautifulSoup pour lire les pages HTML
from bs4 import BeautifulSoup

# ouverture de la page HTML
with open(page_html, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f)
    # extraction de la légende 'alt' de toutes les balises <img> de la page web où se trouve les noms des animaux
    animaux = [animal['alt'] for animal in soup.find_all('img', alt=True)]

print(f"Nombre d'animaux: {len(animaux)}")

print(f"Nom des 5 premiers 'animaux' listés : {animaux[:5]}")
print(f"Nom des 5 derniers 'animaux' listés : {animaux[-5:]}")


In [ ]:
# reformulation du motif, on ignore le premier item de la liste qui est "logo liste animaux" et non un animal
motif_animal_complet = re.compile(rf"({'|'.join(animaux[1:])})", flags=re.IGNORECASE)
print(motif_animal_complet)


def contient_animal_complet(texte):
    return True if re.search(motif_animal_complet, texte) else False

# tous les épigrammes devraient renvoyer True
for texte in data['text_fr']:
    pred = contient_animal_complet(texte)
    print(pred)
    if pred == False:
        print(f"L'épigramme suivant ne mentionne pas d'animal : {texte}")



Avez-vous des idées pour rendre encore plus robuste (c'est à dire capable de couvrir tous les scénarios possibles) ce programme ?

Quelles sont les limites de la logique déductives ?


## Système inductif

Nous allons modéliser très rapidement une logique inductive pour réaliser cette classification basique.

Pour cela il nous faut des exemples d'épigrammes qui ne mentionnent **pas** d'animal. Le repo contient le document `epigrammes_no_animal.csv` qu'il faut ajouter manuellement sur l'environnement virtuel Colab.


In [ ]:
# ajout du corpus d'épigrammes ne mentionnant pas d'animal

try:
    data_no_animal = pd.read_csv(f"{os.getcwd()}/corpus/epigrammes_no_animal.csv")
except FileNotFoundError:
    !wget -P corpus https://raw.githubusercontent.com/alexiaschn/dhsi-2026/main/notebooks/corpus/epigrammes_anthologia_graeca_fr.csv
    data_no_animal = pd.read_csv(f"{os.getcwd()}/corpus/epigrammes_no_animal.csv")


# ajout d'une colonne d'annotation pour chacun des corpus
data_no_animal['animal'] = [False for epigramme in data_no_animal]

data['animal'] = [True for epigramme in data]

# concaténation des deux jeux de données
data = pd.concat([data, data_no_animal])
# à droite : nouvelle colonne 'animal' contenant une valeur booléenne (True/False)
data

In [ ]:
# On va créer un dictionnaire pour stocker les "habitudes" des deux catégories (True or False).
# Structure : { mot : { "True": 2, "False": 4, } }
# chaque mot obtient un compteur d'apparition : si le mot "valise" est présent dans une épigramme classée "True" alors "valise" ajoutera un point à son score "True".

modele_appris = {}

# On parcourt toutes les phrases d'entraînement
for index, epigramme in data.iterrows():
    true_or_false = str(epigramme["animal"])

    # On découpe la phrase en mots (en minuscule pour simplifier)
    mots = epigramme['text_fr'].lower().split()

    for mot in mots:
        # Si le mot n'est pas encore dans le modèle, on l'ajoute
        if mot not in modele_appris:
            modele_appris[mot] = {"True": 0, "False": 0}

        # On "apprend" : on incrémente le compteur de la catégorie du mot
        modele_appris[mot][true_or_false] += 1


In [ ]:

# Affichage de ce que l'algorithme a "appris"
print("--- Modèle appris (Fréquence des mots par catégorie) ---")
for mot, scores in modele_appris.items():
    print(f"Mot '{mot}' : {scores}")


In [ ]:

# 3. LA PHASE DE PRÉDICTION (Classification d'une nouvelle phrase)
def classifier_nouvelle_phrase(texte_nouveau):
    mots = texte_nouveau.lower().split()
    scores_prediction = {"True": 0, "False": 0}

    for mot in mots:
        # Si le mot a été vu pendant l'apprentissage
        if mot in modele_appris:
            # On ajoute les scores appris pour ce mot
            for categorie in scores_prediction:
                scores_prediction[categorie] += modele_appris[mot][categorie]
        else:
            # Si le mot est inconnu, on le traite comme neutre (ou on ignore)
            # Pour simplifier, on ignore ici.
            pass

    # On retourne la catégorie avec le score le plus élevé
    categorie_gagnante = max(scores_prediction, key=scores_prediction.get)
    return categorie_gagnante, scores_prediction

# 4. TESTS
print("\n--- Tests de classification ---")
phrases_tests = [
    "Tu es un beau chat.",
    "Les plumes de cet animal chatoient.",
    "La facture a été payée"
]

for phrase in phrases_tests:
    categorie, scores = classifier_nouvelle_phrase(phrase)
    print(f"Phrase : '{phrase}'")
    print(f"-> Prédiction : {categorie} (Scores : {scores})")

Les modèles inductifs ne sont pas par définition opaques : un accès aux données d'entrainement et à la logique de mise en relation du document avec la catégorie peut rendre transparente la prédiction.



In [ ]:
# Essayer avec d'autres phrases

votre_phrase = input("Entrez votre phrase à classer")

categorie, scores = classifier_nouvelle_phrase(votre_phrase)
print(f"-> Prédiction : {categorie} (Scores : {scores})")


In [ ]:
def classifier_nouvelle_phrase_detailed(texte_nouveau):
    mots = texte_nouveau.lower().split()
    scores_prediction = {"True": 0, "False": 0}

    # 2. Stockage des contributions par mot (NOUVEAU)
    contributions_par_mot = {}

    for mot in mots:
        if mot in modele_appris:
            # On met à jour le score global
            for categorie in scores_prediction:
                scores_prediction[categorie] += modele_appris[mot][categorie]

            # On enregistre combien ce mot a contribué à chaque catégorie
            contributions_par_mot[mot] = modele_appris[mot].copy()
        else:
            contributions_par_mot[mot] = {"True": 0, "False": 0}

    # On trouve le gagnant
    categorie_gagnante = max(scores_prediction, key=scores_prediction.get)

    # On retourne TOUT : la catégorie, le score global, et le détail par mot
    return categorie_gagnante, scores_prediction, contributions_par_mot

# --- TEST AVEC DÉTAIL ---
print("\n--- Analyse détaillée des contributions ---")
phrase_test = "Le chien est un animal domestique. Les poires sont des fruits."
cat, scores, details = classifier_nouvelle_phrase_detailed(phrase_test)

print(f"Phrase : '{phrase_test}'")
print(f"Prédiction finale : {cat} (Score total : {scores})")
print("\nContribution de chaque mot (qui a fait pencher la balance) :")
print(details)
# On trie les mots par "impact" (différence entre le score max et min pour ce mot)
# ou simplement on affiche les scores bruts pour voir les "gros" nombres.
for mot, scores_mot in details.items():
    if mot != "": # On ignore les espaces vides si nécessaire
        # On calcule l'impact : la différence entre la catégorie gagnante et les autres
        impact = scores_mot[cat] - max([v for k, v in scores_mot.items() if k != cat])

        # Affichage formaté
        print(f"  - Mot '{mot}' : {scores_mot}")
        if scores_mot[cat] > 0:
            print(f"      -> A apporté {scores_mot[cat]} point(s) à '{cat}'")
        else:
          print(f"        -> N'était pas présent dans les données d'entrainement.")